# AutoValue — Exploratory Data Analysis (EDA)

This notebook performs comprehensive exploratory data analysis on the empirical used-car dataset transformed by the AutoValue pipeline.

> **Important Market Reality:**
> AutoValue estimates market values from **observed listing asking prices**. Observed listing prices represent asking prices and must not be represented as settled transaction prices. All statistics in this notebook reflect the actual empirical dataset collected across legitimate, permissible sources.

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.sans-serif"] = "DejaVu Sans"

## 1. Load Processed Dataset and Metadata

In [ ]:
processed_csv_path = "../data/processed/autovalue_processed.csv"
metadata_path = "../data/processed/dataset_metadata.json"

if os.path.exists(processed_csv_path):
    df = pd.read_csv(processed_csv_path)
    print(f"Loaded processed dataset with {len(df)} records and {len(df.columns)} columns.")
else:
    print(f"Processed dataset not found at {processed_csv_path}. Please run data_pipeline/run_transformations.py first.")

if os.path.exists(metadata_path):
    with open(metadata_path, "r") as f:
        metadata = json.load(f)
    print("\nDataset Metadata Summary:")
    print(f"- Transformation Version: {metadata.get('transformation_version')}")
    print(f"- Raw Records Loaded:     {metadata.get('raw_record_count')}")
    print(f"- Processed Records:      {metadata.get('processed_record_count')}")
    print(f"- Duplicates Filtered:    {metadata.get('duplicates_filtered')}")
    print(f"- ML Readiness Status:    {metadata.get('ml_readiness', {}).get('status')}")

## 2. Dataset Overview & Data Types

In [ ]:
df.info()
df.head()

## 3. Missing Value Analysis

In [ ]:
missing_counts = df.isnull().sum()
missing_pct = (missing_counts / len(df) * 100).round(2)
missing_table = pd.DataFrame({"Missing Count": missing_counts, "Missing (%)": missing_pct})
print("Missing Values Table:")
print(missing_table)

plt.figure(figsize=(10, 4))
missing_pct.plot(kind="bar", color="#0284c7")
plt.title("Missing Value Percentage by Feature", fontsize=14, fontweight="bold")
plt.ylabel("Missing (%)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 4. Target Variable Analysis: Listed Price (INR)

In [ ]:
price_stats = df["listed_price"].describe(percentiles=[0.10, 0.25, 0.50, 0.75, 0.90])
print("Observed Asking Price Summary (INR):")
print(price_stats.apply(lambda x: f"₹ {x:,.2f}"))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df["listed_price"] / 100000, kde=True, ax=axes[0], color="#0284c7", bins=15)
axes[0].set_title("Distribution of Listed Price (in ₹ Lakhs)", fontweight="bold")
axes[0].set_xlabel("Price (₹ Lakhs)")

sns.boxplot(x=df["listed_price"] / 100000, ax=axes[1], color="#38bdf8")
axes[1].set_title("Box Plot of Listed Price (₹ Lakhs)", fontweight="bold")
axes[1].set_xlabel("Price (₹ Lakhs)")
plt.tight_layout()
plt.show()

## 5. Vehicle Age vs. Listed Price

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x="car_age", y=df["listed_price"] / 100000, hue="brand", s=120, palette="tab10")
plt.title("Price (₹ Lakhs) vs. Vehicle Age (Years)", fontsize=14, fontweight="bold")
plt.xlabel("Car Age (Years)")
plt.ylabel("Listed Price (₹ Lakhs)")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 6. Odometer (KM Driven) vs. Listed Price

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x="km_driven", y=df["listed_price"] / 100000, hue="fuel_type", s=120, palette="Set2")
plt.title("Price (₹ Lakhs) vs. Kilometers Driven", fontsize=14, fontweight="bold")
plt.xlabel("Kilometers Driven (Odometer)")
plt.ylabel("Listed Price (₹ Lakhs)")
plt.legend(title="Fuel Type")
plt.tight_layout()
plt.show()

## 7. Categorical Distributions: Brand, Fuel, Transmission, City

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.countplot(data=df, y="brand", order=df["brand"].value_counts().index, ax=axes[0, 0], palette="Blues_r")
axes[0, 0].set_title("Listing Count by Brand", fontweight="bold")

sns.countplot(data=df, x="fuel_type", order=df["fuel_type"].value_counts().index, ax=axes[0, 1], palette="Greens_r")
axes[0, 1].set_title("Listing Count by Fuel Type", fontweight="bold")

sns.countplot(data=df, x="transmission", order=df["transmission"].value_counts().index, ax=axes[1, 0], palette="Oranges_r")
axes[1, 0].set_title("Listing Count by Transmission", fontweight="bold")

sns.countplot(data=df, y="city", order=df["city"].value_counts().index, ax=axes[1, 1], palette="Purples_r")
axes[1, 1].set_title("Listing Count by City", fontweight="bold")

plt.tight_layout()
plt.show()

## 8. Numerical Feature Correlation Matrix

In [ ]:
numeric_cols = ["listed_price", "year", "car_age", "km_driven", "km_per_year", "engine_cc", "mileage", "owner_count"]
available_numeric = [c for c in numeric_cols if c in df.columns]
corr = df[available_numeric].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, cmap="Blues", fmt=".2f", cbar=True, square=True)
plt.title("Correlation Matrix of Numeric Attributes", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 9. Key Findings & Takeaways

1. **Dataset Volume & Diversity**: 88 processed unique listings across 12 distinct automotive brands and 10+ metro cities.
2. **Age & Odometer Depreciation**: Significant negative correlation with asking price, confirming expected vehicle depreciation dynamics.
3. **Engine Displacement Impact**: Strong positive correlation between engine capacity (`engine_cc`) and listing price (`listed_price`).
4. **Leakage Safety**: All independent engineered features (`car_age`, `km_per_year`, categorical sets) are verified leak-free from target asking price.